
IHM comercial:
- vías técnicas
- vías comerciales
- mismas vías téc con ctc
- mismas vías técnicas y comerciales
- Hay alguna fiable??

In [ ]:
import xml.etree.ElementTree as ET
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from src.api import cargarHistorico
from src.processor import XSIVProcessor
from src.utils import (
    aggregateCounters,
    getEstacionamientos,
    getFilesByDate,
    getPercentageTrue,
    guardarExcel,
    guardarExcelMulti,
    isEmpty,
    isValidCode,
    loadEstaciones,
    map_ctc_subdireccion,
    parallelizeFunction,
    rellenarId,
    sortStrNumbers,
    splitDataframe,
    time2localtime,
)

In [ ]:
ambitos = pd.read_csv("data/Ámbitos.csv", sep=";")
map_ambito = {
    f"{el:0>5}": row[-1]
    for row in ambitos[["RI", "RF", "Ámbito"]].values
    for el in range(row[0], row[1] + 1)
}

In [ ]:
estaciones = loadEstaciones()
# cod2ctc = dict(estaciones[["Código", "NombreCTC"]].values)
cod2ctc = (
    estaciones[["Código", "NombreCTC"]]
    .groupby("Código")
    .agg(lambda x: "|".join(sorted(x)))["NombreCTC"]
    .to_dict()
)
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Platform": "ALTA",
    "PlatformForecast": "PREVISIÓN",
}


mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}

### Análisis vías técnicas PLANIF-CTC

#### trackTable

In [ ]:
tfile = Path("data/trackTable_v01.xml")
with tfile.open("r", encoding="utf8") as f:
    track_txt = f.read()
track_info = ET.fromstring(track_txt)

tracks = {}
for cp in track_info:
    station_id = cp.attrib["controlPointCode"]
    tracks[station_id] = {"vias_planif": [], "vias_xpec": []}
    for track in cp:
        via_planif = track.attrib["name"]
        via_xpec = track.attrib["trackCode"]
        tracks[station_id]["vias_planif"].append(via_planif.strip(" ("))
        tracks[station_id]["vias_xpec"].append(via_xpec.strip(" ("))
tracks = {k: v for k, v in tracks.items() if isValidCode(k)}

In [ ]:
tracks["02030"]

#### MOW

In [ ]:
start_date = "2026-01-19"
end_date = "2026-01-23"
ntrenes = [rellenarId(el) for el in np.arange(100000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    [],
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha"]
).reset_index(drop=True)

In [ ]:
historico_pro[historico_pro["Código"] == "02030"]

In [ ]:
df_logs_ctc = (
    historico_pro.loc[
        (historico_pro["Movimiento"].isin(["LLEGADA", "SALIDA"]))
        & (historico_pro["FuenteVía"] == "CTC"),
        ["NTécnico", "Código", "Nombre", "Vía", "TipoVía", "Fecha"],
    ]
    # .dropna()
    # .drop_duplicates()
).copy()
# df_logs_ctc = df_logs_ctc[df_logs_ctc["Código"].apply(isValidCode)]
# df_logs_ctc = df_logs_ctc.rename(columns={"Vía": "vias_ctc"})


In [ ]:
test = df_logs_ctc[df_logs_ctc["Código"] == "60000"].copy()


In [ ]:
test

In [ ]:
test[test["NTécnico"] == "38563"]

In [ ]:
df_logs_ctc["TipoVía"].unique() 

#### XSIV

In [ ]:
# start_date = "2025-05-28"
# end_date = "2025-05-29"
# dir_logs = Path(r"C:\Users\jose.espinosa\Documents\Data\xsiv\PRO")
# processor = XSIVProcessor()
# w_logs = getFilesByDate(dir_logs, start_date, end_date)
# fnames, full_days = list(zip(*w_logs))
# days = f"{full_days[0].strftime('%Y-%m-%d')} - {full_days[-1].strftime('%Y-%m-%d')}"
# log_list = parallelizeFunction(
#     processor.loadLogFile,
#     data=list(set(fnames)),
#     train_types=train_types,
#     train_operator=None,
#     leave=True,
#     desc=f"Cargando logs: {days}",
# )

In [ ]:
# df_logs_ctc = pd.concat(log_list)[
#     [
#         "movementType",
#         "platform_source",
#         "controlPoint_registerType",
#         "controlPoint_pointCode",
#         "controlPoint_pointName",
#         "platform_platformCode",
#         "timestamp",
#     ]
# ].rename(
#     columns={
#         "platform_source": "fuente",
#         "controlPoint_registerType": "tipo",
#         "controlPoint_pointCode": "código",
#         "controlPoint_pointName": "nombre",
#         "platform_platformCode": "vias_ctc",
#     }
# )

# df_logs_ctc = (
#     df_logs_ctc[
#         (df_logs_ctc["movementType"].isin(["trainArrival", "trainDeparture"]))
#         & (df_logs_ctc["fuente"] == "CTC")
#     ]
#     .dropna()
#     .drop_duplicates()
# )
# df_logs_ctc = df_logs_ctc.drop("movementType", axis=1)[
#     df_logs_ctc["código"].apply(lambda x: bool(regex.search(r"\d{5}", x)))
# ]

#### Vías topos

In [ ]:
estacionamientos_ctc = getEstacionamientos()
# estacionamientos_ctc.groupby(["Código", "MnemónicoComercial"]).agg(
#     {
#         "VíaTécnica": lambda x: sortStrNumbers(set(x)),
#         "Vía": lambda x: sortStrNumbers(set(x)),
#         "Red": lambda x: ",".join(set(x)),
#     }
# )  # ["via_tecnica"]
estacionamientos_ctc = (
    estacionamientos_ctc.groupby(["Código"])
    .agg({"VíaTécnica": sortStrNumbers, "Vía": sortStrNumbers})[
        # "VíaTécnica",
        "Vía"
    ]
    .to_dict()
)

In [ ]:
estacionamientos_ctc["02030"]

#### Merge

In [ ]:
xsiv_not_track = set(df_logs_ctc["Código"].values) - tracks.keys()
track_not_xsiv = tracks.keys() - set(df_logs_ctc["Código"].values)
common = tracks.keys() & set(df_logs_ctc["Código"].values)

In [ ]:
df_logs_ctc[df_logs_ctc["Código"] == "02030"]

In [ ]:
df_logs_ctc["Total"] = 1
xsiv_not_track = (
    df_logs_ctc.loc[
        (df_logs_ctc["Código"].isin(xsiv_not_track))
        & df_logs_ctc["Nombre"].apply(lambda x: "apd" in x.lower()),
        ["Código", "Nombre", "vias_ctc", "Total"],
    ]
    .dropna()
    .groupby(["Código", "Nombre"])
    .agg({"vias_ctc": lambda x: sortStrNumbers(set(x)), "Total": "sum"})
    .reset_index()
    .sort_values(by="Total", ascending=False)
)

track_not_xsiv = (
    pd.DataFrame.from_dict({el: tracks[el] for el in track_not_xsiv}, orient="index")
    .reset_index()
    .rename(columns={"index": "Código"})
)
track_not_xsiv[["vias_planif", "vias_xpec"]] = track_not_xsiv[
    ["vias_planif", "vias_xpec"]
].map(lambda x: sortStrNumbers(set(x)))

common = (
    df_logs_ctc.loc[
        (df_logs_ctc["Código"].isin(common)),
        ["Código", "Nombre", "vias_ctc", "Total"],
    ]
    .dropna()
    .groupby(["Código", "Nombre"])
    .agg({"vias_ctc": lambda x: sortStrNumbers(set(x)), "Total": "sum"})
    .reset_index()
)


In [ ]:
common["vias_topo"] = common["Código"].apply(estacionamientos_ctc.get)

In [ ]:
tracks_df = pd.DataFrame.from_dict(tracks, orient="index")
tracks_df["vias_planif"] = tracks_df["vias_planif"].apply(set).apply(sortStrNumbers)
tracks_df.loc[:, "vias_xpec"] = tracks_df.apply(
    lambda x: sortStrNumbers(set(x["vias_xpec"])), axis=1
)


In [ ]:
tracks_df = tracks_df.reset_index().rename(columns={"index": "Código"})

In [ ]:
vias_tech = pd.merge(
    common,
    tracks_df[["Código", "vias_planif", "vias_xpec"]],
    how="inner",
    on="Código",
)[["Código", "Nombre", "vias_planif", "vias_xpec", "vias_ctc", "vias_topo", "Total"]]

# Combinaciones de nombres de vías que coinciden
cs = ["vias_planif", "vias_xpec", "vias_ctc", "vias_topo"]
new_cols = []
for i1, i2 in combinations(cs, 2):
    c = f"mismas_vias_{i1.strip('vias_')}_{i2.strip('vias_')}"
    new_cols.append(c)
    vias_tech[c] = vias_tech[i1] == vias_tech[i2]

vias_tech["CTC"] = vias_tech["Código"].apply(cod2ctc.get)
vias_tech = vias_tech.sort_values(by="Total", ascending=False)[
    [
        "CTC",
        "Código",
        "Nombre",
        "vias_planif",
        "vias_xpec",
        "vias_ctc",
        "vias_topo",
        "Total",
        # "mismas_vias_xpec_ctc",
        # "mismas_vias_xpec_topo",
        # "mismas_vias_ctc_topo",
        # "mismas_vias_xpec_planif",
        # "mismas_vias_ctc_planif",
    ]
    + new_cols
]

In [ ]:
detalles = []
cols = ["CTC", "Código", "Nombre", "vias_planif", "vias_xpec", "vias_ctc", "vias_topo"]
for _, row in vias_tech.iterrows():
    info = {c: row[c] for c in cols}
    for i1, i2 in combinations(cs, 2):
        s1 = row[i1]
        s1 = set(s1) if s1 else set()
        s2 = row[i2]
        s2 = set(s2) if s2 else set()
        i1 = i1.strip("vias_")
        i2 = i2.strip("vias_")
        info[f"{i1}_{i2}"] = "\n\n".join(
            [
                f"{i1}_no{i2}: {sortStrNumbers(s1 - s2)}",
                f"{i2}_no{i1}: {sortStrNumbers(s2 - s1)}",
            ]
        )
    detalles.append(info)
detalles = pd.DataFrame(detalles)

In [ ]:
vias_tech[vias_tech["Código"] == "02030 "]

In [ ]:
from pathlib import Path
fname = Path(r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\Elcano Calidad Dato\_Análisis Calidad Datos MSE y MIE\03. vías (PLANIF -MSE - IHM COMERCIAL)\23_01_2026_Análisis vía técnicas.xlsx")
dfs = {
    "CTC_Planif": vias_tech,
    "Detalles": detalles,
    # "XSIV_noPlanif": xsiv_not_track,
    # "Planif_noXSIV": track_not_xsiv,
}
guardarExcelMulti(dfs,fname)

In [ ]:
# guardarExcel(
#     vias_tech,
#     Path("Análisis vías técnicas PLANIF-CTC.xlsx"),
#     sheet_name="CTC y Planif",
#     append_sheet=True,
# )
# guardarExcel(
#     xsiv_not_track,
#     Path("Análisis vías técnicas PLANIF-CTC.xlsx"),
#     sheet_name="XSIV no en Planif",
#     append_sheet=True,
# )
# guardarExcel(
#     track_not_xsiv,
#     Path("Análisis vías técnicas PLANIF-CTC.xlsx"),
#     sheet_name="Planif no en XSIV",
#     append_sheet=True,
# )